# Working with numerical data - Part II

This notebook accompanies the script <strong><span style="color:red;">04_Numpy_partB.pdf</span></strong>  and provides practical examples related to its content.

In [ ]:
import numpy as np

<hr style="border: none; height: 20px; background-color: green;">

# 1. Numerical data types

- Python integers store only the numeric value, not the base 
- The base is used only when writing the literal
- Therefore, Python always prints integers in decimal

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Integers in Python and NumPy

### Python ints: arbitrary precision

Python ints support arbitrary precision, so even extremely large integers remain valid int objects (unlike NumPy integers, which have fixed bit‑width)

In [ ]:
x = 10
y = 10**100
print(type(x), type(y))
print(y)

### Integers in any base system will print as base 10 integers

In [ ]:
x = 0b1010   # Binary input (0b = binary)
y = 0o12     # Octal input (0o = octal)
z = 0xA      # Hexadecimal input (0x = hex)

print(x, y, z)   # Output: 10 10 10 (printed in decimal)

#### If you want to display the original base, you must explicitly convert them using `bin()`, `oct()`, or `hex()`

In [ ]:
print(bin(x))   # 0b1010
print(oct(y))   # 0o12
print(hex(z))   # 0xa

### Binary Representation:NumPy `int8`

**NumPy** uses fixed-size, machine-oriented data types.   
That enables predictable memory usage and fast vectorized operations, but it also means **overflow** can happen for integers and **rounding** is unavoidable for floats.

In [ ]:
print("NumPy int8(-24):", np.binary_repr(np.int8(-24), width=8))
print("NumPy int8(24) :", np.binary_repr(np.int8(24), width=8))

### NumPy’s Flexible Integer Type: `np.int_`

A platform-dependent NumPy integer type that matches the native C long type

In [ ]:
arr = np.array([1, 2, 3])    # Uses np.int_

print(arr.dtype)
print(np.dtype(np.int_))
print(arr.dtype == np.dtype(np.int_))

### NumPy Integer Limits with `np.iinfo()`:

Return the minimum and maximum values for a given NumPy integer type

#### Fixed-Width Integer Types

In [ ]:
for t in [np.int8, np.uint8, np.int16, np.uint16, np.int32, np.uint32, np.int64, np.uint64]:
    info = np.iinfo(t)
    print(f"{t.__name__:7}: min={info.min:22} max={info.max:22}")

#### Platform-Dependent Integer Types

In [ ]:
for t in [np.int_, np.uint, np.intp, np.uintp]:
    info = np.iinfo(t)
    print(f"{t.__name__:7}: min={info.min:22} max={info.max:22}")

### Integer overflow in NumPy

Fixed-width integers wrap around on overflow.



In [ ]:
int16_max = np.int16(np.iinfo(np.int16).max)
print("int16 max:", int16_max)
print("max + 1:  ", np.int16(int16_max + 1))

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Float types

NumPy float types follow IEEE 754 (e.g., `float32`, `float64`).
- rounding errors are normal
- overflows produce `inf` unless configured otherwise


In [ ]:
0.1 + 0.2

#### Inspect IEEE-754 bit pattern for `float32`

We can reinterpret the bytes of a `float32` as an unsigned 32-bit integer and print its bit string.

The use of the `struct` module to extract these bits is **only for illustration** and is **not part of the required course material or exam content.**

In [ ]:
import struct

def float32_bits(x: np.float32) -> str:
    bits = struct.unpack("!I", struct.pack("!f", float(x)))[0]
    return format(bits, "032b")  # sign|exponent|mantissa

for v in [np.float32(1.0), np.float32(-5.5), np.float32(1.316554e-36)]:
    print(float32_bits(v), v)

### `np.finfo`

`np.finfo` provides properties like epsilon, min/max normal, and smallest positive subnormal.


#### Fixed-Width Float Types

In [ ]:
for t in [np.float16, np.float32, np.float64]:
    info = np.finfo(t)
    print(f"{t.__name__:7}: min={info.min:24} max={info.max:24} eps={info.eps:24}")

#### Platform-Dependent Float Types

**Note:**
- The alias `np.float_` was removed in the NumPy 2.0 release.
- It was equivalent to `np.float64`.

In [ ]:
for t in [np.longdouble]:
    info = np.finfo(t)
    print(f"{t.__name__:7}: min={info.min:24} max={info.max:24} eps={info.eps:24}")

### Distance to the Next Representable Float

`np.nextafter(x, y)` returns the next representable floating-point number after x in the direction of y.   
It allows us to measure the smallest possible increment between two floating-point values.

In [ ]:
next_float = np.nextafter(np.float32(1), +np.inf) 
next_float - np.float32(1)  

In [ ]:
next_float = np.nextafter(np.float32(1_000_000_000), +np.inf)
next_float - np.float32(1_000_000_000)

### Floating Point Overflows in NumPy

In [ ]:
x = np.float32(np.finfo(np.float32).max)
print("float32 max:", x)
print("max * 2:  ", np.float32(x * 2))  # inf

### Handling Floating-Point Overflows in NumPy

NumPy’s `np.seterr` allows control over how floating-point errors such as overflow are handled, for example by raising an exception instead of silently returning `inf`.

In [ ]:
from pprint import pprint # Pretty-print dictionaries and other data structures

old_settings = np.seterr(over='raise')

a = np.float32(1e38)
b = np.float32(1e38)

try:
    c = a * b
except FloatingPointError as e:
    print("Floating-point overflow detected:", e)

np.seterr(**old_settings) # Restore previous NumPy error handling

print("\nNumPy error settings (restored):")
pprint(np.geterr())

### Floating-Point Overflows in Python
Python does not issue a warning when a floating‑point overflow occurs.

In [ ]:
1e308 * 1e308

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Complex Numbers in NumPy


### Overflow in Complex Numbers

Complex numbers use floating-point values for both the real and imaginary parts.   
If these values exceed the representable range, they overflow independently and become `inf`, as shown in the example.

In [ ]:
z = np.complex128(1e308 + 1e308j)  # Near float64 max
z_overflow = z * 10  # Causes overflow

print(z_overflow)  # Output: (inf+infj)

<hr style="border: none; height: 20px; background-color: green;">

## 2. Comparisons, boolean masks, boolean logic

NumPy comparisons are element-wise and return boolean arrays. Boolean arrays can be used to filter data.

### Vectorized comparisons



In [ ]:
# Both arrays have the same shape
x = np.array([1, 2, 3, 4, 5])
y = np.array([5, 4, 3, 2, 1])

# Operator syntax that internally calls the same ufunc
print("x == y:", x == y)
print("x != y:", x != y)
print("x < y: ", x < y)
print("x >= y:", x >= y)

In [ ]:
# Explicit NumPy ufunc call
print("x == y:",np.equal(x, y))
print("x != y:",np.not_equal(x, y))
print("x < y: ",np.less(x, y))
print("x >= y:", np.greater_equal(x, y))

<hr style="border: none; height: 10px; background-color: LightBlue;">

### When to Prefer NumPy’s Comparison Function

#### When you need the `out=` parameter

In [ ]:
result = np.empty_like(x, dtype=bool)
np.equal(x, y, out=result)

#### When you want method chaining

Method chaining allows you to apply multiple `array` transformations.   
Each transformation is written as a method call.   
Each method operates directly on the result of the previous one.   

In [ ]:
z = np.equal(x, y).astype(int)

In [ ]:
x = np.array([-1, 2, 3, 4, 16])
y = np.array([5, -4, 3, 2, 1])

mask = (x > 2) & (y % 2 == 1)   # non-trivial mask

out = np.full_like(x, fill_value=-1, dtype=np.int8)

z = (
    np.equal(x, y, where=mask, out=out)  # selective comparison into preallocated array
      .astype(np.int8)                   # convert bool → int8 (True→1, False→0)
      .reshape(-1)                       # flatten
      .clip(0, 1)                        # clamp values
      .astype(float)                     # convert to float
)
z

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Bitwise vs. Boolean Operators in Python

#### Python ints (bitwise on single numbers)

In [ ]:
a = 0b1100  # 12 in binary: 00001100
b = 0b1010  # 10 in binary: 00001010

print(a & b)   # 00001000 (bitwise AND)
print(a | b)   # 00001110 (bitwise OR)
print(a ^ b)   # 00000110 (bitwise XOR)
print(~a)      # 11110011 (bitwise NOT, two’s complement)

#### NumPy arrays (element‑wise bitwise operations)

NumPy overloads Python’s bitwise operators (`&`, `|`, `^`, `~`) so they operate element-wise on arrays instead of only on scalar integers.

In [ ]:
arr_1 = np.array([0b1100, 0b1010], dtype=np.int8)  # [12, 10] 
arr_2 = np.array([0b1010, 0b1100], dtype=np.int8)  # [10, 12] 

print(arr_1 & arr_2)  # [00001000, 00001000] -> [8 8] 
print(arr_1 | arr_2)  # [00001110, 00001110] -> [14 14] 
print(arr_1 ^ arr_2)  # [00000110, 00000110] -> [6 6]  
print(~arr_1)         # [11110011, 11110101] -> [243 245]

The same operations are also available as functions such as `np.bitwise_and`, `np.bitwise_or`, ...

In [ ]:
arr_1 = np.array([0b1100, 0b1010], dtype=np.uint8)  # [12, 10] 
arr_2 = np.array([0b1010, 0b1100], dtype=np.uint8)  # [10, 12] 

print(np.bitwise_and(arr_1, arr_2))  # [00001000, 00001000] -> [8 8] 
print(np.bitwise_or(arr_1, arr_2))   # [00001110, 00001110] -> [14 14] 
print(np.bitwise_xor(arr_1, arr_2))  # [00000110, 00000110] -> [6 6]  
print(np.bitwise_not(arr_1))         # [11110011, 11110101] -> [243 245]

<hr style="border: none; height: 10px; background-color: LightBlue;">

### Operator Precedence Pitfall

Bitwise operators are evaluated before comparisons.  
Always use parentheses with `&`, `|`, `^` when combining comparisons.

In [ ]:
a = 5
b = 10

In [ ]:
# This one behaves as expected
if a > 0 and b < 2:
    print("True")
else:
    print("False")

In [ ]:
# This one does not
if a > 0 & b < 2:
    print("True")
else:
    print("False")

In [ ]:
# Always use parentheses when working with bitwise operators
if (a > 0) & (b < 2):
    print("True")
else:
    print("False")

### Operator Precedence Matters

Because NumPy follows bitwise operator precedence, mask_1 and mask_2 evaluate differently unless parentheses are used.

In [ ]:
age  = np.array([15, 72, 30])
chol = np.array([150, 220, 210])
sex  = np.array([0,   0,   0])  # 0 = female

mask_1 = ((age < 18) | (age > 70)) & (chol > 200) & (sex == 0)
mask_2 = (age < 18) | (age > 70) & (chol > 200) & (sex == 0)

print(mask_1)
print(mask_2)
print(np.sum(mask_1), np.sum(mask_2))

<hr style="border: none; height: 10px; background-color: LightBlue;">

### Using Comparisons for Boolean Masks and Filtering in NumPy

#### Comparisons return Boolean arrays

The second array is a scalar, so NumPy will broadcast it across the first array.  

In [ ]:
arr = np.array([2, 4, 1, 5, 9, 0])
mask = arr > 2  # broadcast as if it were [2, 2, 2, 2, 2, 2]

mask

Now we use the Boolean array as a mask to select elements

In [ ]:
arr[mask]

The Boolean mask can also be applied inline

In [ ]:
arr[arr > 2]

#### Combined with multiple conditions

Create a boolean mask for values strictly between 2 and 6

In [ ]:
# Direct boolean indexing (condition applied inline)
print("array   :", arr)
print("selected:", arr[(arr > 2) & (arr < 6)])

In [ ]:
# Two-step approach: create boolean mask, then apply it
mask = (arr > 2) & (arr < 6)
print("array   :", arr)
print("mask    :", mask)
print("selected:", arr[mask])

<hr style="border: none; height: 10px; background-color: LightBlue;">

### Loading Kaggle Datasets into NumPy

#### Reading the CSV File

In [ ]:
import pandas as pd

df = pd.read_csv("../data/csv/heart.csv")

#### Inspecting the First Rows

In [ ]:
df.head()

#### Converting Columns to NumPy Arrays

In [ ]:
chol = df["chol"].to_numpy(copy=True)
age  = df["age"].to_numpy(copy=True)
sex  = df["sex"].to_numpy(copy=True)

#### Hint: Read‑Only Arrays in NumPy
In NumPy, arrays can also exist in a non‑writable (read‑only) form when the underlying data should not be modified.

In [ ]:
arr_1 = df["chol"].to_numpy()
arr_2 = df["chol"].to_numpy(copy=True)

print(f"Is arr_1 writeable? {arr_1.flags.writeable}")
print(f"Is arr_2 writeable? {arr_2.flags.writeable}")

# Attempt to modify arr_1
try:
    arr_1[0] = 0
except ValueError:
    print("arr_1 is non‑writable (read‑only)")

# This one works
arr_2[0] = 0
print("arr_2 successfully modified")


#### Comparisons return Boolean arrays

In [ ]:
bool_arr = chol > 200
print(bool_arr)

#### Using the Boolean array as a mask

In [ ]:
filtered_chol_values = chol[chol > 200]
print(filtered_chol_values[:10])

#### Combining Multiple Conditions

In [ ]:
mask = (chol > 200) & (chol < 210)
filtered_chol_values = chol[mask]
print(filtered_chol_values[:10])

In [ ]:
mask = (chol > 200) & (age > 50)
filtered_chol_values = chol[mask]
print(filtered_chol_values[:10])

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Boolean arrays for counting entries

Using `np.count_nonzero()` counts **how many values** satisfy a condition:

In [ ]:
np.count_nonzero(chol < 180)

Since True is considered as 1 and False as 0, we can sum up True 

In [ ]:
np.sum(chol < 180)

#### Get the percentage of entries matching a condition

In [ ]:
np.mean(chol < 180) * 100

#### Unique counts of categorical values (e.g., sex distribution)

In [ ]:
unique_values, counts = np.unique(sex, return_counts=True)
print(f"unique_values: {unique_values}")
print(f"counts: {counts}")

In [ ]:
percentages = counts / counts.sum() * 100

for value, count, pct in zip(unique_values, counts, percentages):
    print(f"Sex {value}: {count} entries ({pct:.2f}%)")

### Checking Conditions with `np.any()` and `np.all()`

- `np.any(mask)` checks if at least one is True
- `np.all(mask)` checks if all are True

In [ ]:
# checks if at least one is True
print(f"Any patients with less cholesterol than 180 mg/dl? -> {np.any(chol < 180)}") 
print(f"Any patients with less cholesterol than 100 mg/dl? -> {np.any(chol < 100)}") 

# checks if all are True
print(f"Do all patients have more cholesterol than 100 mg/dl? -> {np.all(chol > 100)}")
print(f"Do all patients have more cholesterol than 180 mg/dl? -> {np.all(chol < 180)}") 

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Some more Boolean Masks Examples

Identify high‑risk combinations Shows how multiple conditions combine with AND‑logic to detect specific subgroups.


In [ ]:
np.sum(((age > 70) & (chol > 300) & (sex == 1)))

Mix AND/OR logic in one mask Demonstrates operator precedence and the use of parentheses in complex filters.

In [ ]:
np.sum(((age < 18) | (age > 70)) & (chol > 200) & (sex == 0))

In [ ]:
np.sum((age < 18) | (age > 70) & (chol > 200) & (sex == 0))

Count cases with at least one missing value Illustrates OR‑logic across variables to detect incomplete observations.

In [ ]:
np.sum((np.isnan(chol) | np.isnan(age) | np.isnan(sex)))

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Boolean Mask on a 2D Array

In [ ]:
arr = np.array([
    [180, 155],
    [220, 148],
    [195, 263],
    [240, 152],
    [205, 145]
])

mask = arr > 200
print("mask:\n", mask)

filtered_values = arr[mask]
print("\nfiltered values (>200):")
print(filtered_values)

### Row Filtering with a 1D Boolean Mask
Build a 1D mask from column conditions (e.g., chol > 200 and age > 50) to select entire rows.   
Columns must have distinct meanings, as in our dataset (chol, age, sex).

In [ ]:
arr_2d = np.column_stack((chol, age, sex))

chol_col = arr_2d[:, 0]   # chol column
age_col  = arr_2d[:, 1]   # age column

mask = (chol_col > 200) & (age_col > 50)

filtered = arr_2d[mask]
print(filtered[:4])

<hr style="border: none; height: 20px; background-color: green;">

## 3. Broadcasting

Default: We use arrays of the same size. Binary ufunc operations are element-by-element.  
**Broadcasting:** Binary ufunc operations can also be performed on arrays of different sizes due to broadcasting.  




In [ ]:
# By default, we use arrays of the same size
arr_1 = np.array([0, 1, 2])
arr_2 = np.array([5, 5, 5]) 
print(arr_1 + arr_2)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Broadcasting: Scalar + Array

In [ ]:
# Use Broadcasting. Adding a scalar
arr_1 = np.array([0, 1, 2])
arr_1 = arr_1 + 5
print(arr_1)

In [ ]:
arr_1 = np.array([0, 1, 2])
arr_1 += 5
print(arr_1)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Broadcasting: 1D + 2D Array

Extending broadcasting to arrays of higher dimensions.  
The one-dimensional array `a` is broadcasted across the second dimension to match the shape of `b`

In [ ]:
arr_1 = np.ones((3, 3))
print(arr_1)

In [ ]:
arr_2 = np.array([0, 1, 2])
print(arr_1 + arr_2)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Broadcasting: 1D + 2D (Both Sides Stretch)


In some cases, dimensions from both arrays conceptually change so that the operation becomes possible.   

In [ ]:
arr_1 = np.array([[0],
              [1],
              [2]])
arr_2 = np.array([0, 1, 2])

print(arr_1 + arr_2)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Examples of broadcasting

### Example I

We add a 1D array to a 2D array. Broadcasting pads the 1D array with a leading 1 and then stretches it.

In [ ]:
arr_1 = np.ones((2, 3))
arr_2 = np.arange(3)

print("arr_1.shape:", arr_1.shape)
print("arr_2.shape:", arr_2.shape)

In [ ]:
print(arr_1 + arr_2)

### Example II

We add a column vector `(3,1)` to a row vector `(3,)`. Both are broadcast to `(3,3)`.

In [ ]:
arr_2 = np.arange(3)
arr_1 = arr_2.reshape((3, 1))

print("arr_1.shape:", arr_1.shape)
print("arr_2.shape:", arr_2.shape)

In [ ]:
print(arr_1 + arr_2)

### Example III

Here broadcasting fails because the final shapes would need to be `(3,2)` and `(3,3)`, which violates Rule 3.

In [ ]:
arr_1 = np.ones((3, 2))
arr_2 = np.arange(3)

print("arr_1.shape:", arr_1.shape)
print("arr_2.shape:", arr_2.shape)

In [ ]:
try:
    print(arr_1 + arr_2)
except ValueError as e:
    print("ValueError:", e)

<hr style="border: none; height: 20px; background-color: green;">

# 4. Fancy indexing and combined indexing


## How can we access multiple elements of an array at once?

### Different options:

#### Option: NOT fancy indexing



In [ ]:
arr = np.arange(12)

np.array([arr[3], arr[7], arr[2]])

#### Option: Fancy indexing


In [ ]:
# Fancy indexing
ind = [3, 7, 2] 
arr[ind]     

In [ ]:
arr[[3, 7, 2]]     

### Fancy Indexing in 2D

#### Example: Selecting rows

In [ ]:
arr = np.arange(12).reshape(4, 3)

arr[[0, 2]]

#### Example: Selecting columns

In [ ]:
arr[:, [0, 2]]

#### Example: row indices + column indices

In [ ]:
arr[[0, 2], [1, 2]]

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Combined indexing

NumPy allows combining different indexing methods such as slicing, fancy indexing (integer arrays), and boolean masks within the same indexing operation.

This makes it possible to filter rows and select specific columns at the same time, which is common when working with multi-dimensional arrays.

In [ ]:
arr = np.arange(30).reshape(10, 3)
print(arr)

### row filter + column selection

In [ ]:
arr[arr[:, 0] > 15, 1]

#### row filter + multiple columns

In [ ]:
arr[arr[:, 0] > 15, 1:3]

#### specific rows + column

In [ ]:
arr[[1, 5, 8], 2]

#### slice rows + specific columns

In [ ]:
arr[2:6, [0, 2]]